# E19 关联规则：购物篮支持度 / 置信度 / 提升度

> 覆盖矩阵：关联规则（纯 Python，无 mlxtend 强依赖）

## 你将学到
1. 事务数据编码；支持度、置信度、提升度。
2. 高提升度 ≠ 可干预因果。
3. **缺口**：包内无关联规则 API（宜放可选 extras）。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 合成购物篮

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
items = ['bread', 'milk', 'eggs', 'butter', 'diaper', 'beer']
baskets = []
for i in range(200):
    b = set(rng.choice(items[:4], size=rng.integers(1, 4), replace=False))
    # 注入模式：diaper → beer
    if rng.random() < 0.25:
        b |= {'diaper', 'beer'}
    baskets.append(frozenset(b))
# 长表便于 Analyzer
rows = [{'txn': i, 'item': it} for i, b in enumerate(baskets) for it in b]
df = pd.DataFrame(rows)
print(df.head(), 'n_txn', df.txn.nunique())
print(DataAnalyzer(df).get_basic_info().get('shape'))


## 2. 频繁 1-/2-项集与规则

In [ ]:
from collections import Counter
from pathlib import Path

n_txn = len(baskets)
min_support = 0.08
c1 = Counter()
for b in baskets:
    for it in b:
        c1[frozenset([it])] += 1
freq1 = {k: v / n_txn for k, v in c1.items() if v / n_txn >= min_support}
c2 = Counter()
for b in baskets:
    for a, c in combinations(sorted(b), 2):
        c2[frozenset([a, c])] += 1
freq2 = {k: v / n_txn for k, v in c2.items() if v / n_txn >= min_support}
rules = []
for pair, sup in freq2.items():
    a, b = tuple(pair)
    for ant, con in ((a, b), (b, a)):
        ant_f = frozenset([ant])
        if ant_f not in freq1:
            continue
        conf = sup / freq1[ant_f]
        lift = conf / freq1[frozenset([con])]
        rules.append(dict(antecedent=ant, consequent=con, support=sup, confidence=conf, lift=lift))
rules_df = pd.DataFrame(rules).sort_values('lift', ascending=False)
print(rules_df.head(10))
out = Path('outputs/19_association_rules'); out.mkdir(parents=True, exist_ok=True)
rules_df.to_csv(out / 'rules.csv', index=False)


## 附录
- 作业：解释 diaper→beer 高 lift 仍可能是混杂；设计门店实验。
- 包优化：可选 `AssociationRules.mine(transactions, min_support=...)`。
